# AnthropicToolCallState

```lua
stateDiagram-v2
    INIT --> CHAT
    CHAT --> FINAL
```

```mermaid
stateDiagram-v2
    Direction LR
    INIT --> CHAT
    CHAT --> FINAL
```


### a) Normal Chat Flow

In [1]:
from gai.asm import AsyncStateMachine

with AsyncStateMachine.StateMachineBuilder(
    """
    INIT --> CHAT
    CHAT--> FINAL
    """
) as builder:
    fsm = builder.build(
        {
            "INIT": {
                "input_data": {
                    "llm_config": {"type": "getter", "dependency": "get_llm_config"},
                }
            },
            "CHAT": {
                "module_path": "gai.asm.states",
                "class_name": "AnthropicChatState",
                "title": "CHAT",
                "input_data": {
                    "llm_config": {"type": "state_bag", "dependency": "llm_config"},
                },
                "output_data": ["streamer", "get_assistant_message"],
            },
            "FINAL": {
                "output_data": ["monologue"],
            },
        },
        get_llm_config=lambda state: {
            "client_type": "anthropic",
            #"model": "claude-opus-4-20250514",
            "model": "claude-sonnet-4-20250514",
            "max_tokens": 32000,
            "temperature": 0.7,
            "top_p": 0.95,
        }
    )

## Step 2: INIT --> TOOL_CALL

fsm.user_message = "Tell me a one sentence story."
await fsm.run_async()
async for chunk in fsm.state_bag["streamer"]:
    if (isinstance(chunk,str)):
        print(chunk, end='', flush=True)
print("\n\n")

## Step 4: Print the state history
for message in fsm.state_history[1]["output"]["monologue"].list_messages():
    print(
        f"{message.header.timestamp} {message.header.sender} > {message.body.content}"
    )

The last person on Earth sat alone in a room, when suddenly there was a knock at the door.


1752495084.0183284 User > Tell me a one sentence story.
1752495088.0559342 Assistant > [{'citations': None, 'text': 'The last person on Earth sat alone in a room, when suddenly there was a knock at the door.', 'type': 'text'}]


### b) Tool Call Flow

In [2]:
from gai.asm import AsyncStateMachine
from gai.mcp.client import McpAggregatedClient

with AsyncStateMachine.StateMachineBuilder(
    """
    INIT --> CHAT
    CHAT--> FINAL
    """
) as builder:
    fsm = builder.build(
        {
            "INIT": {
                "input_data": {
                    "llm_config": {"type": "getter", "dependency": "get_llm_config"},
                    "mcp_client": {"type": "getter", "dependency": "get_mcp_client"},
                }
            },
            "CHAT": {
                "module_path": "gai.asm.states",
                "class_name": "AnthropicChatState",
                "title": "CHAT",
                "input_data": {
                    "llm_config": {"type": "state_bag", "dependency": "llm_config"},
                    "mcp_client": {"type": "state_bag","dependency": "mcp_client"},
                },
                "output_data": ["streamer", "get_assistant_message"],
            },
            "FINAL": {
                "output_data": ["monologue"],
            },
        },
        get_llm_config=lambda state: {
            "client_type": "anthropic",
            #"model": "claude-opus-4-20250514",
            "model": "claude-sonnet-4-20250514",
            "max_tokens": 32000,
            "temperature": 0.7,
            "top_p": 0.95,
        },
        get_mcp_client=lambda state: McpAggregatedClient(["mcp-time"])
    )

## Step 2: INIT --> CHAT

fsm.user_message = "What time is it in Singapore?"
await fsm.run_async()
async for chunk in fsm.state_bag["streamer"]:
    if (isinstance(chunk,str)):
        print(chunk, end='', flush=True)
print("\n\n")

## Step 4: Print the state history
for message in fsm.state_history[1]["output"]["monologue"].list_messages():
    print(
        f"{message.header.timestamp} {message.header.sender} > {message.body.content}"
    )

I'll get the current time in Singapore for you.


1752495098.6496954 User > What time is it in Singapore?
1752495101.1412172 Assistant > [{'citations': None, 'text': "I'll get the current time in Singapore for you.", 'type': 'text'}, {'id': 'toolu_01JeYMBX1bbKxqUSqpPkNSj5', 'input': {'format': 'YYYY-MM-DD HH:mm:ss', 'timezone': 'Asia/Singapore'}, 'name': 'current_time', 'type': 'tool_use'}]
